# Rust 110: Structs, OOP and Traits — Practice

Companion exercises for `10_structs-oop-and-traits_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. Struct Anatomy: Three Forms and Their Constructors](#1-struct-anatomy-three-forms-and-their-constructors)
- [2. Impls: Associated Functions, Methods and `Self`](#2-impls-associated-functions-methods-and-self)
- [3. Visibility: `pub` and the Privacy Boundary](#3-visibility-pub-and-the-privacy-boundary)
- [4. Derive: Free Impls from One Line](#4-derive-free-impls-from-one-line)
- [5. Newtypes: Distinct Types at Zero Cost](#5-newtypes-distinct-types-at-zero-cost)
- [6. Traits: Contracts with Default Methods](#6-traits-contracts-with-default-methods)
- [7. Generics: One Definition, Many Types](#7-generics-one-definition-many-types)
- [8. Trait Objects: `dyn` and Dynamic Dispatch](#8-trait-objects-dyn-and-dynamic-dispatch)
- [9. Enums with Data: The Sum Type](#9-enums-with-data-the-sum-type)
- [10. Operators Are Traits: `Add`, `Display` and `Debug`](#10-operators-are-traits-add-display-and-debug)
- [11. Static vs Dynamic Dispatch: The Cost Model](#11-static-vs-dynamic-dispatch-the-cost-model)
- [12. Layout: Padding, Field Order and `repr`](#12-layout-padding-field-order-and-repr)
- [13. Reading the Compiler: Panics and E-Codes](#13-reading-the-compiler-panics-and-e-codes)
- [15. Review: Mixed Problems](#15-review-mixed-problems)

§1–§13 each have one exercise per demonstration cell of the example notebook; §15 mixes them. When a check fails, the fastest route back is §15 of the example notebook, which is the cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A `fn` or `struct` may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Three notes specific to this chapter

- **Given code is complete and stays.** Many cells hand you a struct, enum or trait so the stub has something to implement. Do not rename given items — later cells reuse them deliberately (`Order` from §1 returns in §8 and §15; `Qty` from §5 returns in §10; `Summarize` from §6 returns in §7). This is the one-namespace rule working *for* you.
- **E-codes are the syllabus.** Several exercises are calibrated so the wrong approach produces the exact diagnostic the example notebook catalogues in §13 — E0034, E0038, E0117, E0605. When you see one, you are one step from the lesson.
- **No files, no crates.** Everything here is std-only and in-process; there is no scratch directory to clean.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines that serve a stub you have not solved yet;
- `unused`/`never used` items — a few given traits and methods exist so you can answer a question *about* them (§8.2) without calling them.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 and §13 of the example notebook.


### Exercise 0.1 — The smoke test

**Goal**: prove the kernel compiles your items and runs your code, before any real exercise.

**Objectives**

- implement `kernel_pulse` to return the chapter's number
- keep the signature unchanged

**See**: §0.1 of the example notebook.

**Hint**: the body is one tail expression; no semicolon.


In [ ]:
/// Return the chapter number — the smallest possible proof the kernel is alive.
///
/// TODO: replace `todo!()` with the chapter number.
fn kernel_pulse() -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(kernel_pulse(), 10);
println!("Exercise 0.1 passed");


### Exercise 1.1 — Build an order

**Goal**: complete the two methods that make a named struct usable: a constructor and a reader.

**Given**: the `Order` struct — `symbol: String`, `qty: u32`, `px_cents: u32` — is fully declared; only the method bodies are missing.

**Objectives**

- implement `new` so it produces an `Order` from the three parts (the parameter names already match the fields — use the shorthand where it applies)
- implement `notional` to return `qty * px_cents / 100` as `f64` dollars
- keep the signatures unchanged

**See**: §1.1 and §2.1 of the example notebook.

**Hint**: the constructor is one struct literal; `symbol.to_string()` bridges `&str` to `String`, and `as f64` bridges the integers to dollars.


In [ ]:
#[derive(Debug)]
struct Order {
    symbol: String,
    qty: u32,
    px_cents: u32,
}
impl Order {
    /// Build an order from its parts.
    ///
    /// TODO: replace `todo!()` with the struct literal.
    fn new(symbol: &str, qty: u32, px_cents: u32) -> Self {
        todo!()
    }
    /// Notional value in dollars.
    ///
    /// TODO: replace `todo!()` with the arithmetic — qty * px_cents / 100.
    fn notional(&self) -> f64 {
        todo!()
    }
}

// --- check (do not modify) ---
let o = Order::new("AAPL", 300, 19_251);
assert_eq!(o.qty, 300);
assert_eq!(o.px_cents, 19_251);
assert_eq!(o.notional(), 57753.0);
println!("Exercise 1.1 passed");


### Exercise 1.2 — Functional update without re-typing

**Goal**: derive a changed config from a base with struct-update syntax, and see which fields the base loses.

**Given**: the `Config` struct — `name: String`, `ticks: u32`, `verbose: bool`.

**Objectives**

- implement `with_ticks` to return a `Config` with `ticks` replaced and everything else taken from `base`
- keep the signature unchanged
- note what the move rule does to `base`: it moved **by value** into the function (chapter 05), and inside, `..base` moved its `String` out of the local — after your call, nothing of the caller's `base` is readable

**See**: §1.2 of the example notebook.

**Hint**: the body is one struct literal: the new `ticks`, then `..base`.


In [ ]:
#[derive(Debug, PartialEq)]
struct Config {
    name: String,
    ticks: u32,
    verbose: bool,
}
impl Config {
    /// Same config, one field changed — functional update, not a re-typed literal.
    ///
    /// TODO: replace `todo!()` with a struct literal using `..base`.
    fn with_ticks(base: Config, ticks: u32) -> Config {
        todo!()
    }
}

// --- check (do not modify) ---
let base = Config { name: String::from("feed"), ticks: 10, verbose: false };
let tuned = Config::with_ticks(base, 50);
assert_eq!(tuned, Config { name: String::from("feed"), ticks: 50, verbose: false });
// Move accounting: `base` moved by value into the call, and `..base` moved its
// String field out of the local inside — reading `base.name` after the literal
// in the function body is E0382, the diagnostic §1.2 catalogued.
println!("Exercise 1.2 passed");


### Exercise 2.1 — The three receivers

**Goal**: implement one method per receiver form and watch each one change what the caller keeps.

**Given**: the `Ticket` struct — `symbol: String`, `qty: u32`.

**Objectives**

- implement `new` (associated function), `notional` (`&self`), `scale` (`&mut self`), and `into_parts` (`self`, consumes)
- keep the signatures unchanged

**See**: §2.1 of the example notebook.

**Hint**: `scale` mutates in place with `*`-assignment; `into_parts` returns a tuple built from both fields, which is what consumes `self`.


In [ ]:
#[derive(Debug)]
struct Ticket {
    symbol: String,
    qty: u32,
}
impl Ticket {
    /// Associated function: no receiver — a constructor.
    ///
    /// TODO: replace `todo!()` with the struct literal.
    fn new(symbol: &str, qty: u32) -> Self {
        todo!()
    }
    /// &self: read-only borrow.
    ///
    /// TODO: replace `todo!()` with qty * px as f64.
    fn notional(&self, px: f64) -> f64 {
        todo!()
    }
    /// &mut self: in-place update.
    ///
    /// TODO: replace `todo!()` with the multiplication.
    fn scale(&mut self, k: u32) {
        todo!()
    }
    /// self: consumes the value, field-by-field.
    ///
    /// TODO: replace `todo!()` with the (symbol, qty) tuple.
    fn into_parts(self) -> (String, u32) {
        todo!()
    }
}

// --- check (do not modify) ---
let mut t = Ticket::new("ES", 10);
assert_eq!(t.notional(4126.5), 41265.0);
t.scale(3);
assert_eq!(t.qty, 30);
let (symbol, qty) = t.into_parts();
assert_eq!((symbol.as_str(), qty), ("ES", 30));
println!("Exercise 2.1 passed");


### Exercise 2.2 — Field and method, same name

**Goal**: give `Holding` a `qty` method alongside its `qty` field, and prove the parentheses decide which is meant.

**Given**: the `Holding` struct — `qty: u32`, `symbol: String` — is fully declared.

**Objectives**

- implement `qty` (the method) to return the field's value
- implement `label` to produce `"{symbol} x{qty}"` using the field
- keep the signatures unchanged

**See**: §2.2 of the example notebook.

**Hint**: inside the method, `self.qty` with no parentheses is the field; `format!` assembles the label.


In [ ]:
#[derive(Debug)]
struct Holding {
    qty: u32,
    symbol: String,
}
impl Holding {
    /// A method named exactly like the field: legal, and the parens decide.
    ///
    /// TODO: replace `todo!()` with the field's value.
    fn qty(&self) -> u32 {
        todo!()
    }
    /// Readable one-line rendering.
    ///
    /// TODO: replace `todo!()` with the formatted label.
    fn label(&self) -> String {
        todo!()
    }
}

// --- check (do not modify) ---
let f = Holding { qty: 100, symbol: String::from("ES") };
let field = f.qty;
let method = f.qty();
assert_eq!(field, 100);
assert_eq!(method, field);
assert_eq!(f.label(), "ES x100");
println!("Exercise 2.2 passed");


### Exercise 3.1 — An invariant behind a constructor

**Goal**: make a zero-starting account whose balance can only grow by positive deposits — and prove the invariant panics when attacked.

**Given**: the `Account` struct — `balance_cents: i64` — is fully declared.

**Objectives**

- implement `new` (balance zero), `deposit` (panic on non-positive amounts, add otherwise), and `balance` (the reader)
- keep the signatures unchanged
- the check catches your panic with `catch_unwind` — no payload extraction needed, `is_err` is enough

**See**: §3.1 of the example notebook and §1.3 of chapter 08.

**Hint**: `assert!(cents > 0, "deposit must be positive");` is the whole guard; the message text is checked by the grader's panic — keep it exact.


In [ ]:
#[derive(Debug)]
struct Account {
    balance_cents: i64,
}
impl Account {
    /// A fresh account starts at zero.
    ///
    /// TODO: replace `todo!()` with the struct literal.
    fn new() -> Self {
        todo!()
    }
    /// Add a positive amount; panic on anything else.
    ///
    /// TODO: replace `todo!()` with the guard plus the addition.
    fn deposit(&mut self, cents: i64) {
        todo!()
    }
    /// The reader: the only way to see the balance.
    ///
    /// TODO: replace `todo!()` with the field.
    fn balance(&self) -> i64 {
        todo!()
    }
}

// --- check (do not modify) ---
let mut acct = Account::new();
assert_eq!(acct.balance(), 0);
acct.deposit(10_000);
acct.deposit(2_500);
assert_eq!(acct.balance(), 12_500);
let refused = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| {
    let mut a = Account::new();
    a.deposit(-1);
}));
assert!(refused.is_err(), "negative deposit must panic");
println!("Exercise 3.1 passed");


### Exercise 4.1 — Derives at work

**Goal**: use the derived `Eq` + `Hash` (and nothing else) to count distinct prices.

**Given**: the `Price` struct with `#[derive(Debug, Clone, PartialEq, Eq, Hash)]` — every derive you need is already in the line.

**Objectives**

- implement `count_distinct` to return how many unique prices the slice holds
- keep the signature unchanged
- do not add any derive — the given line is already correct, and `Hash` is what makes the set possible

**See**: §4.1 of the example notebook.

**Hint**: `prices.iter().collect::<HashSet<&Price>>()` builds a set of borrows; its `.len()` is the answer.


In [ ]:
use std::collections::HashSet;
#[derive(Debug, Clone, PartialEq, Eq, Hash)]
struct Price {
    symbol: String,
    cents: u32,
}
/// Count unique prices — Eq + Hash do the work, field order decides the equality.
///
/// TODO: replace `todo!()` with a HashSet-based count.
fn count_distinct(prices: &[Price]) -> usize {
    todo!()
}

// --- check (do not modify) ---
let feed = [
    Price { symbol: String::from("AAPL"), cents: 19_251 },
    Price { symbol: String::from("MSFT"), cents: 41_500 },
    Price { symbol: String::from("AAPL"), cents: 19_251 },
];
assert_eq!(count_distinct(&feed), 2);
assert_eq!(count_distinct(&[]), 0);
println!("Exercise 4.1 passed");


### Exercise 5.1 — The newtype with a gate

**Goal**: build `Qty` — a `u32` that cannot be zero, prints itself, and costs nothing.

**Given**: the tuple struct `Qty(u32)` with `Debug, Clone, Copy, PartialEq, Eq, PartialOrd, Ord` derived.

**Objectives**

- implement `new` to return `Some(Qty(raw))` for any positive raw count and `None` for zero
- implement `value` to expose the inner count
- implement `Display` so a `Qty` renders as `"{n} lots"`
- keep the signatures unchanged

**See**: §5.1 of the example notebook.

**Hint**: `write!(f, "{} lots", self.0)` is the whole `Display` body; the `fmt` imports are already at the top of the cell.


In [ ]:
use std::fmt::{self, Display};
#[derive(Debug, Clone, Copy, PartialEq, Eq, PartialOrd, Ord)]
struct Qty(u32);
impl Qty {
    /// The single constructor: zero is illegal, so it returns None.
    ///
    /// TODO: replace `todo!()` with the guard-and-wrap.
    fn new(raw: u32) -> Option<Self> {
        todo!()
    }
    /// The inner count.
    ///
    /// TODO: replace `todo!()` with the field.
    fn value(self) -> u32 {
        todo!()
    }
}
impl Display for Qty {
    /// TODO: replace `todo!()` with the write! — "{n} lots".
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        todo!()
    }
}

// --- check (do not modify) ---
assert!(Qty::new(0).is_none());
let q = Qty::new(300).expect("300 is legal");
assert_eq!(q.value(), 300);
assert_eq!(format!("{q}"), "300 lots");
assert_eq!(size_of::<Qty>(), size_of::<u32>());
println!("Exercise 5.1 passed");


### Exercise 5.2 — The zero-cost marker

**Goal**: pass a `Closed` marker where a `&str` would have been magic — and prove it weighed nothing.

**Given**: the unit struct `Closed` with `Debug` and `Default` derived.

**Objectives**

- implement `gate` to accept a `Closed` value (ignore it with `_`) and return the fixed message `"market closed; orders rejected"`
- keep the signature unchanged

**See**: §5.2 of the example notebook.

**Hint**: the parameter is written `_: Closed`; the body is one string literal.


In [ ]:
#[derive(Debug, Default)]
struct Closed;
/// The type IS the permission: only a `Closed` value unlocks this message.
///
/// TODO: replace `todo!()` with the fixed message.
fn gate(_: Closed) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(gate(Closed), "market closed; orders rejected");
assert_eq!(gate(Closed::default()), "market closed; orders rejected");
assert_eq!(size_of::<Closed>(), 0);
println!("Exercise 5.2 passed");


### Exercise 6.1 — Implement the contract

**Goal**: satisfy the `Summarize` trait for two types — one riding the default `summary`, one overriding it.

**Given**: the `Summarize` trait (required `headline`, default `summary` wrapping it in `[feed]`), plus the structs `Trade { symbol, qty }` and `Halt { symbol }`.

**Objectives**

- implement `headline` for `Trade` as `"{symbol} x{qty}"`
- implement `headline` for `Halt` as `"HALT {symbol}"` **and** override `summary` to wrap in `[risk]` instead
- keep every signature unchanged

**See**: §6.1 of the example notebook.

**Hint**: `format!("{} x{}", self.symbol, self.qty)` is the shape; the override is just a second `fn summary` in `Halt`'s impl block.


In [ ]:
trait Summarize {
    /// The one required method: the contract's core.
    fn headline(&self) -> String;
    /// Provided: every implementor gets this unless it opts out.
    fn summary(&self) -> String {
        format!("[feed] {}", self.headline())
    }
}
struct Trade {
    symbol: String,
    qty: u32,
}
struct Halt {
    symbol: String,
}
impl Summarize for Trade {
    /// TODO: replace `todo!()` with "{symbol} x{qty}".
    fn headline(&self) -> String {
        todo!()
    }
}
impl Summarize for Halt {
    /// TODO: replace `todo!()` with "HALT {symbol}".
    fn headline(&self) -> String {
        todo!()
    }
    /// The override: Halt reports through the risk desk.
    ///
    /// TODO: replace `todo!()` with the [risk]-wrapped headline.
    fn summary(&self) -> String {
        todo!()
    }
}

// --- check (do not modify) ---
let t = Trade { symbol: String::from("CL"), qty: 10 };
let h = Halt { symbol: String::from("NVDA") };
assert_eq!(t.summary(), "[feed] CL x10");
assert_eq!(h.summary(), "[risk] HALT NVDA");
println!("Exercise 6.1 passed");


### Exercise 6.2 — Disambiguate with UFCS

**Goal**: fill in two same-named trait methods, then write the fully qualified calls the ambiguity forces.

**Given**: the `Quote` struct and two traits, `Named` and `Priced`, both declaring `describe`. The impl bodies are yours; the check block already uses the qualified syntax.

**Objectives**

- implement `Named::describe` to produce `"named {symbol}"` and `Priced::describe` to produce `"priced {cents}"`
- implement `ambiguous` to return both descriptions as a tuple, calling each through its trait
- keep the signatures unchanged

**See**: §6.2 of the example notebook.

**Hint**: `<Quote as Named>::describe(&q)` names the trait explicitly — that syntax works inside `ambiguous` too, once per side of the tuple.


In [ ]:
struct Quote {
    symbol: String,
    cents: u32,
}
trait Named {
    fn describe(&self) -> String;
}
trait Priced {
    fn describe(&self) -> String;
}
impl Named for Quote {
    /// TODO: replace `todo!()` with "named {symbol}".
    fn describe(&self) -> String {
        todo!()
    }
}
impl Priced for Quote {
    /// TODO: replace `todo!()` with "priced {cents}".
    fn describe(&self) -> String {
        todo!()
    }
}
/// Both descriptions, each through its own trait.
///
/// TODO: replace `todo!()` with the pair of qualified calls.
fn ambiguous(q: &Quote) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
let q = Quote { symbol: String::from("AAPL"), cents: 19_251 };
assert_eq!(<Quote as Named>::describe(&q), "named AAPL");
assert_eq!(<Quote as Priced>::describe(&q), "priced 19251");
assert_eq!(
    ambiguous(&q),
    ("named AAPL".to_string(), "priced 19251".to_string())
);
// The bare dot form is E0034 while both traits are in scope:
// q.describe(); // error[E0034]: multiple applicable items in scope
println!("Exercise 6.2 passed");


### Exercise 7.1 — A bounded generic method

**Goal**: write `(low, high)` once for every comparable, copyable `T`.

**Given**: the generic struct `Candle<T>` with `open` and `close` fields. The impl header — `impl<T: Copy + PartialOrd> Candle<T>` — is already written; only the body is missing.

**Objectives**

- implement `range` to return `(min, max)` of the two fields
- keep the signature unchanged
- the bounds do the compile-time checking: `Candle::<String>` has no `range` at all

**See**: §7.1 of the example notebook.

**Hint**: one `if self.open <= self.close` with two tail branches; returning the fields is why the impl needs `Copy`.


In [ ]:
#[derive(Debug, PartialEq)]
struct Candle<T> {
    open: T,
    close: T,
}
impl<T: Copy + PartialOrd> Candle<T> {
    /// Returns (low, high); needs Copy to return the values, PartialOrd to compare.
    ///
    /// TODO: replace `todo!()` with the ordered pair.
    fn range(&self) -> (T, T) {
        todo!()
    }
}

// --- check (do not modify) ---
let c = Candle { open: 4125.0f64, close: 4126.5 };
assert_eq!(c.range(), (4125.0, 4126.5));
let down = Candle { open: 20u32, close: 10 };
assert_eq!(down.range(), (10, 20));
println!("Exercise 7.1 passed");


### Exercise 7.2 — `impl Trait` in argument position

**Goal**: write one function that accepts a slice of *anything* summarizable — and returns `None` for an empty one.

**Given**: `Summarize` and `Trade` from Exercise 6.1 are already defined and in scope (the one namespace at work — run §6.1 first after a kernel restart).

**Objectives**

- implement `loudest` to return the longest `summary()` in the slice, or `None` when it is empty
- keep the signature unchanged — `&[impl Summarize]` is the whole generic machinery

**See**: §7.2 of the example notebook.

**Hint**: `items.iter().map(|i| i.summary()).max_by_key(|s| s.len())` is the iterator chain; `max_by_key` already returns an `Option`.


In [ ]:
/// The longest summary in the slice; None when empty.
///
/// TODO: replace `todo!()` with the map + max_by_key chain.
fn loudest(items: &[impl Summarize]) -> Option<String> {
    todo!()
}

// --- check (do not modify) ---
let trades = [
    Trade { symbol: String::from("CL"), qty: 10 },
    Trade { symbol: String::from("HO"), qty: 5 },
];
assert_eq!(loudest(&trades).as_deref(), Some("[feed] CL x10"));
let empty: Vec<Trade> = Vec::new();
assert_eq!(loudest(&empty).as_deref(), None);
println!("Exercise 7.2 passed");


### Exercise 8.1 — Two types behind one contract

**Goal**: box two different types into one `Vec<dyn Report>` and prove the vtable dispatches per element.

**Given**: the dyn-compatible `Report` trait (required `flash`, no consts, no generics), plus `Trade` and `Halt` from Exercise 6.1 — already defined and in scope.

**Objectives**

- implement `roster` to return both implementations in one `Vec<Box<dyn Report>>`, `Trade` first, `Halt` second
- implement `flash_all` to collect every element's `flash()` string in order
- keep the signatures unchanged

**See**: §8.1 of the example notebook.

**Hint**: `vec![Box::new(t) as Box<dyn Report>, Box::new(h)]` — the `as` cast guides the first element's type inference; `flash_all` maps over the vector of boxes.


In [ ]:
/// A dyn-compatible twin of Summarize: no consts, no generics, no self-by-value.
/// Its method is `flash` because Halt already receives `headline` from Summarize,
/// and a second trait method of the same name would make every Halt receiver
/// ambiguous (E0034 — the exact trap §8.2's catalogue teaches).
trait Report {
    fn flash(&self) -> String;
}
impl Report for Trade {
    /// TODO: replace `todo!()` with the same shape as Summarize's headline.
    fn flash(&self) -> String {
        todo!()
    }
}
impl Report for Halt {
    /// TODO: replace `todo!()` with the same shape as Summarize's headline.
    fn flash(&self) -> String {
        todo!()
    }
}
/// Both types behind one contract, Trade first.
///
/// TODO: replace `todo!()` with the vec! of two boxed items.
fn roster(t: Trade, h: Halt) -> Vec<Box<dyn Report>> {
    todo!()
}
/// Every element's flash, in order.
///
/// TODO: replace `todo!()` with the map-collect over the boxes.
fn flash_all(items: &[Box<dyn Report>]) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let mixed = roster(
    Trade { symbol: String::from("GC"), qty: 1 },
    Halt { symbol: String::from("NVDA") },
);
assert_eq!(flash_all(&mixed), vec!["GC x1".to_string(), "HALT NVDA".to_string()]);
assert_eq!(size_of::<&dyn Report>(), 16); // the fat pointer
println!("Exercise 8.1 passed");


### Exercise 8.2 — The dyn-compatibility checklist

**Goal**: fill in the three blocker stubs *without calling them through `dyn`*, then answer the catalogue question with a returned flag.

**Given**: the `Flex` trait (default `show`, a generic `eq_other`) and the `Eater` trait (a `self`-by-value `eat`). These traits exist so you can answer *about* them; direct calls on concrete types work fine — only forming `dyn` is refused.

**Objectives**

- implement the three body stubs (`show_tag`, `eq_other`, `eat`) exactly as their doc comments describe
- implement `dyn_ready` to return the given answer — the trait with a generic method cannot be made into a trait object
- keep the signatures unchanged

**See**: §8.2 of the example notebook.

**Hint**: `false` satisfies `eq_other`'s signature without naming `T`'s operations; the `dyn_ready` answer is the one `bool` in the exercise.


In [ ]:
trait Flex {
    /// Provided default: every implementor shows the same tag.
    fn show(&self) -> String {
        "flex".to_string()
    }
    /// Generic method: blocks `dyn Flex` (a vtable needs one pointer per T).
    fn eq_other<T>(&self, other: T) -> bool {
        let _ = other;
        false
    }
}
struct Flexy;
impl Flex for Flexy {}
trait Eater {
    /// self by value: blocks calls through `&dyn Eater` (E0161/E0507).
    fn eat(self) -> u32;
}
struct Meal;
impl Eater for Meal {
    /// TODO: replace `todo!()` with the fixed count.
    fn eat(self) -> u32 {
        todo!()
    }
}
/// Direct calls on concrete types are fine — dyn is what's refused.
///
/// TODO: replace `todo!()` with the tag.
fn show_tag(f: &Flexy) -> String {
    todo!()
}
/// The catalogue question: can a trait with a generic method be used as dyn?
///
/// TODO: replace `todo!()` with the answer.
fn dyn_ready() -> bool {
    todo!()
}

// --- check (do not modify) ---
let f = Flexy;
assert_eq!(show_tag(&f), "flex");
assert!(!f.eq_other(5u32));
assert_eq!(Meal.eat(), 1);
assert_eq!(dyn_ready(), false);
// The refused forms, for the record:
// let d: &dyn Flex = &Flexy;      // error[E0038]: the trait `Flex` is not dyn compatible
// let e: &dyn Eater = &Meal;      // E0161/E0507 at the call site: d.eat()
println!("Exercise 8.2 passed");


### Exercise 9.1 — The sum type at work

**Goal**: implement `Fill`'s behaviour — a method that reads any variant, a `Default`, and `matches!` on top.

**Given**: the `Fill` enum with `#[derive(Debug, Default, PartialEq)]` and `#[default] New`; the `filled` signature is yours to complete.

**Objectives**

- implement `filled`: `New` fills nothing, `Partial` has filled `qty` so far, `Full` filled all `qty`
- implement `describe` to return `"{n} filled"` for every variant, using `filled`
- keep the signatures unchanged
- the check pins `matches!` as a shape test on your own enum

**See**: §9.1 of the example notebook.

**Hint**: the match has three arms; `Fill::Partial { qty, .. }` ignores `remaining` with the rest pattern.


In [ ]:
#[derive(Debug, Default, PartialEq)]
enum Fill {
    #[default]
    New,
    Partial { qty: u32, remaining: u32 },
    Full { qty: u32 },
}
impl Fill {
    /// Filled quantity so far, whatever the variant.
    ///
    /// TODO: replace `todo!()` with the three-arm match.
    fn filled(&self) -> u32 {
        todo!()
    }
    /// Readable summary: "{n} filled".
    ///
    /// TODO: replace `todo!()` with the formatted count.
    fn describe(&self) -> String {
        todo!()
    }
}

// --- check (do not modify) ---
let fills = [
    Fill::Partial { qty: 300, remaining: 120 },
    Fill::Full { qty: 300 },
    Fill::New,
];
assert_eq!(fills[0].filled(), 300);
assert_eq!(fills[2].filled(), 0);
assert_eq!(fills[0].describe(), "300 filled");
assert_eq!(Fill::default(), Fill::New);
assert!(matches!(fills[1], Fill::Full { .. }));
println!("Exercise 9.1 passed");


### Exercise 9.2 — Discriminants and declaration order

**Goal**: pin an explicit-discriminant enum and keep the two orderings straight: `as` reads the discriminant, derived `Ord` reads declaration order.

**Given**: the `Level` enum with discriminants 1/5/10 and `PartialOrd, Ord` derived; the `magnitude` signature is yours to complete.

**Objectives**

- implement `magnitude` to return each variant's discriminant as a `u8`
- implement `sort_levels` to return the slice's levels in derived-`Ord` order (declaration order)
- keep the signatures unchanged

**See**: §9.2 of the example notebook.

**Hint**: `Level::Mid as u8` is the whole cast — fieldless only; the sort is `v.to_vec()` then `sort()`.


In [ ]:
#[derive(Debug, Clone, Copy, PartialEq, Eq, PartialOrd, Ord)]
enum Level {
    Low = 1,
    Mid = 5,
    High = 10,
}
/// The discriminant, read with `as` — fieldless enums only.
///
/// TODO: replace `todo!()` with the match returning each discriminant.
fn magnitude(level: Level) -> u8 {
    todo!()
}
/// Derived Ord: declaration order decides, not the discriminants.
///
/// TODO: replace `todo!()` with a sorted copy.
fn sort_levels(levels: &[Level]) -> Vec<Level> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(magnitude(Level::Low), 1);
assert_eq!(magnitude(Level::Mid), 5);
assert_eq!(magnitude(Level::High), 10);
let shuffled = [Level::High, Level::Low, Level::Mid];
assert_eq!(sort_levels(&shuffled), vec![Level::Low, Level::Mid, Level::High]);
println!("Exercise 9.2 passed");


### Exercise 10.1 — `Add` for the newtype

**Goal**: make `Qty + Qty`, `Qty - Qty` and `Qty * u32` real — and prove the operands are consumed exactly like chapter 05 taught.

**Given**: `Qty` from Exercise 5.1 is in scope (run §5.1 first after a kernel restart); the `use std::ops::..` line and impl headers are written.

**Objectives**

- implement `Add::add`, `Sub::sub` and `Mul<u32>::mul` on `Qty`, each wrapping in `Qty(..)`
- keep the `type Output = Qty;` lines and signatures unchanged

**See**: §10.1 of the example notebook.

**Hint**: each body is one line: `Qty(self.0 + rhs.0)` — the associated type is already `Qty`, so the wrapper is the whole job.


In [ ]:
use std::ops::{Add, Sub, Mul};
impl Add for Qty {
    type Output = Qty;
    /// TODO: replace `todo!()` with the wrapped sum.
    fn add(self, rhs: Qty) -> Qty {
        todo!()
    }
}
impl Sub for Qty {
    type Output = Qty;
    /// TODO: replace `todo!()` with the wrapped difference.
    fn sub(self, rhs: Qty) -> Qty {
        todo!()
    }
}
impl Mul<u32> for Qty {
    type Output = Qty;
    /// TODO: replace `todo!()` with the wrapped product.
    fn mul(self, k: u32) -> Qty {
        todo!()
    }
}

// --- check (do not modify) ---
let q = Qty(300);
assert_eq!(q + Qty(20), Qty(320));
assert_eq!(q - Qty(100), Qty(200));
assert_eq!(q * 3, Qty(900));
// Qty is Copy, so q survives the operators; with a non-Copy type the first
// `+` would move the left operand — the exact §10.1 lesson.
assert_eq!(q.value(), 300);
println!("Exercise 10.1 passed");


### Exercise 10.2–10.3 — A width-aware `Display`

**Goal**: write the `write!`-based `Display` for `Pct`, then the `f.pad` twin that honours the caller's width flags.

**Given**: the two structs; the `fmt` imports from Exercise 5.1 are still in scope.

**Objectives**

- implement `Pct`'s `Display` with `write!(f, "{:.1}%", self.0 * 100.0)` — note this one *ignores* `{:>width}`
- implement `PadPct`'s `Display` with `f.pad(&format!("{:.1}%", self.0 * 100.0))` — this one *honours* it
- implement `padded` to return `PadPct`'s rendering under `{:>10}`
- keep the signatures unchanged

**See**: §10.2 and §10.3 of the example notebook.

**Hint**: the difference between the two bodies is one line: `write!` vs `f.pad(&format!(..))`; `padded` is `format!("{:>10}", PadPct(v))`.


In [ ]:
struct Pct(f64);
impl Display for Pct {
    /// TODO: replace `todo!()` with the write! — note it ignores width flags.
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        todo!()
    }
}
struct PadPct(f64);
impl Display for PadPct {
    /// TODO: replace `todo!()` with f.pad over the same formatted text.
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        todo!()
    }
}
/// The width-honouring rendering under {:>10}.
///
/// TODO: replace `todo!()` with the format! call.
fn padded(v: f64) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(format!("{}", Pct(0.125)), "12.5%");
assert_eq!(format!("{:>10}", Pct(0.125)), "12.5%"); // width ignored by write!
assert_eq!(format!("{}", PadPct(0.125)), "12.5%");
assert_eq!(padded(0.125), "     12.5%");            // width honoured by f.pad
assert_eq!(format!("{:<10}", PadPct(0.125)), "12.5%     ");
println!("Exercise 10.2 passed");


### Exercise 10.4 — Newtype around a foreign type

**Goal**: give `Vec<u32>` a `Display` the legal way — by wrapping it — since `impl Display for Vec<u32>` directly is E0117.

**Given**: the wrapper `Lots(Vec<u32>)` with its `Display` impl header written.

**Objectives**

- implement `Lots::from_raw` to build the wrapper from a `Vec<u32>`
- implement `Display` to render `"[{n} lots]"` where `{n}` is the vector's length
- implement `total` to return the sum of the wrapped quantities
- keep the signatures unchanged

**See**: §10.4 of the example notebook.

**Hint**: `write!(f, "[{} lots]", self.0.len())` is the rendering; `iter().sum()` totals the inner values; the orphan rule is why the impl lives on `Lots`, not on `Vec`.


In [ ]:
#[derive(Debug)]
struct Lots(Vec<u32>);
impl Lots {
    /// The wrapper constructor — the escape hatch the orphan rule forces.
    ///
    /// TODO: replace `todo!()` with the tuple-struct wrap.
    fn from_raw(values: Vec<u32>) -> Self {
        todo!()
    }
    /// The wrapped quantities, added up.
    ///
    /// TODO: replace `todo!()` with the sum.
    fn total(&self) -> u32 {
        todo!()
    }
}
impl Display for Lots {
    /// TODO: replace `todo!()` with the "[{n} lots]" rendering.
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        todo!()
    }
}

// --- check (do not modify) ---
let lots = Lots::from_raw(vec![100, 200]);
assert_eq!(format!("{lots}"), "[2 lots]");
assert_eq!(lots.total(), 300);
// The refused direct form, for the record:
// impl Display for Vec<u32> { .. } // error[E0117]: only traits defined in the
//                                  // current crate can be implemented for types
//                                  // defined outside of the crate
println!("Exercise 10.4 passed");


### Exercise 11.1 — Same numbers, different time

**Goal**: write the static (generic) and dynamic (`dyn`) totalizers side by side and observe that the results agree — the cost, not the correctness, is what differs.

**Given**: the `Shape` trait (`area`), the `Rec` struct (`w`, `h`), and the two function signatures.

**Objectives**

- implement `static_total` over `&[Rec]` with an iterator chain
- implement `dyn_total` over `&[Box<dyn Shape>]`, same chain, different element type
- implement `boxed_recs` to lift every `Rec` into a `Box<dyn Shape>`
- keep the signatures unchanged

**See**: §11.1 of the example notebook.

**Hint**: `.iter().map(|s| s.area()).sum()` works in both — only the slice's element type differs; `Box::new(r) as Box<dyn Shape>` does the lift.


In [ ]:
trait Shape {
    fn area(&self) -> f64;
}
#[derive(Debug, Clone, Copy)]
struct Rec {
    w: f64,
    h: f64,
}
impl Shape for Rec {
    /// TODO: replace `todo!()` with the area.
    fn area(&self) -> f64 {
        todo!()
    }
}
/// Static dispatch: one monomorphised copy per concrete type.
///
/// TODO: replace `todo!()` with the map-sum chain.
fn static_total(shapes: &[Rec]) -> f64 {
    todo!()
}
/// The same computation through trait objects.
///
/// TODO: replace `todo!()` with the map-sum chain.
fn dyn_total(shapes: &[Box<dyn Shape>]) -> f64 {
    todo!()
}
/// Lift concrete recs into boxed trait objects.
///
/// TODO: replace `todo!()` with the map over recs, boxing each.
fn boxed_recs(recs: &[Rec]) -> Vec<Box<dyn Shape>> {
    todo!()
}

// --- check (do not modify) ---
let recs = vec![Rec { w: 3.0, h: 2.0 }, Rec { w: 4.0, h: 5.0 }];
assert_eq!(static_total(&recs), 26.0);
let boxed = boxed_recs(&recs);
assert_eq!(dyn_total(&boxed), 26.0);
assert_eq!(static_total(&recs), dyn_total(&boxed)); // same answer; the cost differs
println!("Exercise 11.1 passed");


### Exercise 12.1 — The layout table

**Goal**: prove the zero-cost claims with your own hands — the newtype, the unit struct, and the fat pointer.

**Given**: `Qty`, `Closed` and `Report` are in scope from earlier sections (run the notebook top-to-bottom after a kernel restart).

**Objectives**

- implement `cost_of_newtype` to return `size_of::<Qty>()`
- implement `cost_of_marker` to return `size_of::<Closed>()`
- implement `cost_of_dyn_ref` to return `size_of::<&dyn Report>()`
- keep the signatures unchanged

**See**: §12.1 of the example notebook.

**Hint**: each body is one `size_of::<T>()` call — the checks encode the whole table.


In [ ]:
/// The newtype costs exactly what its inner type costs.
///
/// TODO: replace `todo!()` with the size query.
fn cost_of_newtype() -> usize {
    todo!()
}
/// The unit struct costs nothing.
///
/// TODO: replace `todo!()` with the size query.
fn cost_of_marker() -> usize {
    todo!()
}
/// A dyn reference is fat: data pointer + vtable pointer.
///
/// TODO: replace `todo!()` with the size query.
fn cost_of_dyn_ref() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(cost_of_newtype(), size_of::<u32>());
assert_eq!(cost_of_marker(), 0);
assert_eq!(cost_of_dyn_ref(), 16);
println!("Exercise 12.1 passed");


### Exercise 12.2–13.1 — The repr question

**Goal**: pack the same three fields two ways and answer the catalogue question with a returned number instead of a comment.

**Given**: `Packed` (repr(Rust) — the compiler may reorder) and `Frozen` (repr(C) — declaration order stands), same fields.

**Objectives**

- implement `rust_packed` to return `size_of::<Packed>()`
- implement `c_frozen` to return `size_of::<Frozen>()`
- implement `padding_in_frozen` to return the number of padding bytes `Frozen` carries (12 total − 6 bytes of fields)
- keep the signatures unchanged

**See**: §12.1, §12.2 and §13 of the example notebook.

**Hint**: repr(Rust) packs `flag`+`done` together for 8; repr(C) freezes the written order for 12; the padding count is `c_frozen() - 6`.


In [ ]:
#[repr(Rust)]
struct Packed {
    flag: u8,
    qty: u32,
    done: u8,
}
#[repr(C)]
struct Frozen {
    flag: u8,
    qty: u32,
    done: u8,
}
/// repr(Rust): the compiler reorders to minimise padding.
///
/// TODO: replace `todo!()` with the size query.
fn rust_packed() -> usize {
    todo!()
}
/// repr(C): declaration order stands, gaps and all.
///
/// TODO: replace `todo!()` with the size query.
fn c_frozen() -> usize {
    todo!()
}
/// Padding bytes inside Frozen: total minus the 6 bytes of real fields.
///
/// TODO: replace `todo!()` with the subtraction.
fn padding_in_frozen() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(rust_packed(), 8);
assert_eq!(c_frozen(), 12);
assert_eq!(padding_in_frozen(), 6);
// The E-code that guards the discriminant rule (13.1):
// Fill::New as u8; // error[E0605]: non-primitive cast: `Fill` as `u8`
println!("Exercise 12.2 passed");


## 15. Review: Mixed Problems

Two problems that assemble the chapter: a trait hierarchy spanning struct, enum and generic, and a newtype that composes `Display`, operators and an invariant.

### Exercise 15.1 — The summarizer stack


In [ ]:
/// One contract over three shapes: a named struct, a tuple struct, an enum.
trait Bullet {
    fn line(&self) -> String;
}
struct Trade2 {
    symbol: String,
    qty: u32,
}
struct Alert(u32);
#[derive(Debug, PartialEq)]
enum Notice {
    Halt(String),
    Resume,
}
impl Bullet for Trade2 {
    /// TODO: replace `todo!()` with "{symbol} x{qty}".
    fn line(&self) -> String {
        todo!()
    }
}
impl Bullet for Alert {
    /// TODO: replace `todo!()` with "alert level {n}".
    fn line(&self) -> String {
        todo!()
    }
}
impl Bullet for Notice {
    /// TODO: replace `todo!()` with "HALT {s}" for Halt(s), "RESUME" for Resume.
    fn line(&self) -> String {
        todo!()
    }
}
/// Every line, in order, through the trait.
///
/// TODO: replace `todo!()` with the map over boxed items.
fn board(items: &[Box<dyn Bullet>]) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let items: Vec<Box<dyn Bullet>> = vec![
    Box::new(Trade2 { symbol: String::from("CL"), qty: 10 }),
    Box::new(Alert(2)),
    Box::new(Notice::Halt(String::from("NVDA"))),
    Box::new(Notice::Resume),
];
assert_eq!(
    board(&items),
    vec![
        "CL x10".to_string(),
        "alert level 2".to_string(),
        "HALT NVDA".to_string(),
        "RESUME".to_string(),
    ]
);
println!("Exercise 15.1 passed");


### Exercise 15.2 — The money type

**Goal**: build `Cents` — an `i64` newtype whose constructor refuses negatives, whose `+` is real, and whose `Display` is money-shaped.

**Given**: the derive line and the impl/trait skeletons; the `Display` import from Exercise 5.1 is in scope.

**Objectives**

- implement `Cents::new` returning `None` for negative amounts (a price is never negative)
- implement `Add` so `Cents + Cents` is a `Cents`
- implement `Display` rendering `"$1.50"` for 150 — dollars, a point, two digits
- implement `charge` to sum the slice's non-negative total... through the trait ops, i.e. `fold(Cents(0), |acc, c| acc + *c)`
- keep the signatures unchanged

**See**: §5, §10.1 and §10.2 of the example notebook.

**Hint**: `write!(f, "${}.{:02}", self.0 / 100, self.0 % 100)` is the money rendering — integer division and remainder, no floats anywhere.


In [ ]:
#[derive(Debug, Clone, Copy, PartialEq, Eq, PartialOrd, Ord)]
struct Cents(i64);
impl Cents {
    /// The constructor: negative money is unrepresentable.
    ///
    /// TODO: replace `todo!()` with the guard-and-wrap.
    fn new(raw: i64) -> Option<Self> {
        todo!()
    }
}
impl Add for Cents {
    type Output = Cents;
    /// TODO: replace `todo!()` with the wrapped sum.
    fn add(self, rhs: Cents) -> Cents {
        todo!()
    }
}
impl Display for Cents {
    /// TODO: replace `todo!()` with the $d.dd rendering.
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        todo!()
    }
}
/// The bill: sum every charge through the `+` you wrote.
///
/// TODO: replace `todo!()` with the fold.
fn charge(items: &[Cents]) -> Cents {
    todo!()
}

// --- check (do not modify) ---
assert!(Cents::new(-1).is_none());
let a = Cents::new(150).expect("legal");
let b = Cents::new(99).expect("legal");
assert_eq!(a + b, Cents(249));
assert_eq!(format!("{a}"), "$1.50");
assert_eq!(format!("{b}"), "$0.99");
assert_eq!(charge(&[a, b, Cents(1)]), Cents(250));
assert_eq!(charge(&[]), Cents(0));
println!("Exercise 15.2 passed");


### What Comes Next

Chapter 10's types now exist in both forms — taught and tested. The applied track continues:

| Chapter | Title | What it settles |
|---|---|---|
| **11** | **Errors in Depth** | error enums for the `Option`s `Qty::new` and `Cents::new` returned here; `From`/`?`; `Box<dyn Error>` — the error twin of §8's trait objects |
| 12 | Iterators in Depth | the `Iterator` trait from the author's side — the trait §7's bounds and §11's chains quietly used |
| 13 | Modules, Crates and Testing | where `pub` finally bites, and `#[test]` versions of every check block this notebook graded |

**Carry forward.** You should now be able to do these without looking anything up:

- build a type from the three struct forms and gate its construction
- pick the right receiver (`&self`/`&mut self`/`self`) without hesitating
- write a trait with a default method, implement it for several types, and disambiguate with UFCS
- make a type generic with bounds and say when `dyn` is the right call instead
- implement `Add`/`Display` for a newtype — and reach for the wrapper when the orphan rule says so
- recite the size table (newtype = inner, unit = 0, `&dyn` = 16) and the two reprs' field-order rules
